# 08 — Comparaison finale des modèles

**Objectif :** agréger les métriques, visualiser les écarts et sélectionner le modèle final.

**Entrées :** rapports JSON des notebooks 03 à 07.  
**Sorties :** tableau CSV, graphique et décision finale.  
**Dépendance :** notebook 07.  
**Temps estimé :** moins d'une minute.  
**Ressources :** CPU uniquement.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
REPORTS_DIR = ROOT / "reports"

metric_files = [
    "metrics_popularity.json",
    "metrics_query_click.json",
    "metrics_tfidf.json",
    "metrics_semantic.json",
    "metrics_hybrid.json",
]
reports = [json.loads((REPORTS_DIR / name).read_text(encoding="utf-8")) for name in metric_files]
comparison = pd.DataFrame(
    [
        {
            "model": report["model"],
            "map@5": report["map@5"],
            "recall@5": report["recall@5"],
            "mrr@5": report["mrr@5"],
            "hit_rate@5": report["hit_rate@5"],
        }
        for report in reports
    ]
).sort_values(["map@5", "mrr@5"], ascending=False, kind="stable")

display(comparison.style.format({column: "{:.3f}" for column in comparison.columns[1:]}))

## Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
comparison.set_index("model")[["map@5", "recall@5", "mrr@5"]].plot.bar(ax=ax)
ax.set(title="Comparaison des modèles", ylabel="Score", ylim=(0, 1.05), xlabel="")
ax.legend(loc="lower right")
fig.tight_layout()
figure_path = REPORTS_DIR / "figures" / "model_comparison.png"
fig.savefig(figure_path, dpi=150)
plt.close(fig)

## Sélection et limites

In [ ]:
inventory = json.loads((REPORTS_DIR / "data_inventory.json").read_text(encoding="utf-8"))
source = inventory["source"]
best_row = comparison.iloc[0].to_dict()
selection = {
    "selected_model": best_row["model"],
    "selection_metric": "map@5",
    "validation_source": source,
    "metrics": {key: float(value) for key, value in best_row.items() if key != "model"},
    "provisional": source != "kaggle",
    "warning": (
        "Résultats synthétiques : réexécuter tous les notebooks avec les données Kaggle avant "
        "de considérer les scores comme représentatifs."
        if source != "kaggle"
        else "Résultats calculés sur le split local des données Kaggle."
    ),
}

comparison.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
(REPORTS_DIR / "final_model.json").write_text(
    json.dumps(selection, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(selection, indent=2, ensure_ascii=False))

## Conclusion

La décision est explicitement marquée provisoire tant que les fichiers Kaggle ne sont pas
disponibles. Aucun score synthétique ne doit être présenté comme un résultat de compétition.